In [1]:
import subprocess, urllib.request

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(urllib.request.urlopen("https://huggingface.co", timeout=10).status)

Tue Nov  3 17:32:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from pathlib import Path
import shutil

# Repère les fichiers importés dans Kaggle.
source = next(Path("/kaggle/input").rglob("ablation/manifest.json")).parent.parent

# Copie les données et le lanceur dans le dossier de travail.
for name in ("ablation", "pipeline"):
    shutil.copytree(source / name, Path("/kaggle/working") / name, dirs_exist_ok=True)

%cd /kaggle/working
!git clone https://github.com/Meuleur/Piscine.git kit
!git -C kit checkout --detach 3709c85ddaabfcd688d6eea80a5c5ed90605b549
%pip install tokenizers==0.23.3

/kaggle/working
Cloning into 'kit'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 37 (delta 5), reused 37 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 950.53 KiB | 5.22 MiB/s, done.
Resolving deltas: 100% (5/5), done.
HEAD is now at 3709c85 Lien du dépôt : Meuleur/Piscine
Note: you may need to restart the kernel to use updated packages.


In [2]:
%cd /kaggle/working
!python -m pipeline.ablation_runs run --kit kit --prepared ablation --results runs/ablation

/kaggle/working
vérification de l'empreinte du dataset...
modèle : 10.17M paramètres | 3052 pas x 32,768 tokens = 100,007,936 tokens | device cuda:0 torch.float16 | dataset 100,007,937 tokens (1.0 passes)
pas 60/3052 | loss 6.5495 | lr 8.00e-04 | 119.5k tok/s | reste ~14 min
pas 130/3052 | loss 5.4965 | lr 1.73e-03 | 126.9k tok/s | reste ~13 min
pas 190/3052 | loss 5.1559 | lr 2.00e-03 | 128.1k tok/s | reste ~12 min
pas 250/3052 | loss 4.8049 | lr 1.99e-03 | 128.0k tok/s | reste ~12 min
pas 310/3052 | loss 4.7602 | lr 1.99e-03 | 127.3k tok/s | reste ~12 min
pas 370/3052 | loss 4.6693 | lr 1.97e-03 | 126.0k tok/s | reste ~12 min
pas 430/3052 | loss 4.4108 | lr 1.96e-03 | 125.1k tok/s | reste ~11 min
pas 490/3052 | loss 4.1904 | lr 1.94e-03 | 124.9k tok/s | reste ~11 min
pas 500 | BPB validation (1/4 du jeu) : 1.8357
pas 550/3052 | loss 3.8510 | lr 1.92e-03 | 124.6k tok/s | reste ~11 min
pas 610/3052 | loss 4.2152 | lr 1.89e-03 | 124.7k tok/s | reste ~11 min
pas 670/3052 | loss 4.0615 | 

In [3]:
%cd /kaggle/working
!zip ablation-results.zip runs/ablation/manifest.json runs/ablation/environment.json runs/ablation/*/result.json


/kaggle/working
  adding: runs/ablation/manifest.json (deflated 68%)
  adding: runs/ablation/environment.json (deflated 23%)
  adding: runs/ablation/clean/result.json (deflated 59%)
  adding: runs/ablation/dedup/result.json (deflated 59%)
  adding: runs/ablation/raw/result.json (deflated 59%)


In [6]:
from pathlib import Path
import shutil

print("Recherche du dataset…", flush=True)
source = next(
    p.parent for p in Path("/kaggle/input").rglob("manifest.json")
    if (p.parent / "quality_pii").is_dir()
)
print("Dataset trouvé :", source, flush=True)

# Affiche chaque fichier avant et après sa copie.
def copy_with_progress(src, dst):
    print("Copie :", src, flush=True)
    result = shutil.copy2(src, dst)
    print("Copie terminée", flush=True)
    return result

shutil.copytree(
    source,
    "/kaggle/working/ablation-extra",
    dirs_exist_ok=True,
    copy_function=copy_with_progress,
)
shutil.copytree(
    source.parent / "pipeline",
    "/kaggle/working/pipeline",
    dirs_exist_ok=True,
    copy_function=copy_with_progress,
)

print("Préparation terminée", flush=True)

Recherche du dataset…
Dataset trouvé : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/manifest.json
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/pii/data_meta.json
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/pii/train.bin
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/quality/data_meta.json
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/quality/train.bin
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/quality_pii/data_meta.json
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/quality_pii/train.bin
Copie terminée
Copie : /kaggle/input/datasets/matissegarlot/piscine-ablation-extra/ablation/tokenizer.json
Copie terminée
Copi

In [7]:
%cd /kaggle/working
!python -u -m pipeline.ablation_runs run --kit kit --prepared ablation-extra --results runs/ablation-extra

/kaggle/working
vérification de l'empreinte du dataset...
reprise au pas 1500
modèle : 10.17M paramètres | 3052 pas x 32,768 tokens = 100,007,936 tokens | device cuda:0 torch.float16 | dataset 100,007,937 tokens (1.0 passes)
pas 1560/3052 | loss 3.4433 | lr 1.14e-03 | 124.4k tok/s | reste ~7 min
pas 1620/3052 | loss 3.3501 | lr 1.08e-03 | 124.0k tok/s | reste ~6 min
pas 1680/3052 | loss 3.4301 | lr 1.02e-03 | 122.1k tok/s | reste ~6 min
pas 1740/3052 | loss 3.3291 | lr 9.66e-04 | 121.6k tok/s | reste ~6 min
pas 1800/3052 | loss 3.2757 | lr 9.09e-04 | 122.1k tok/s | reste ~6 min
pas 1860/3052 | loss 3.2349 | lr 8.52e-04 | 122.7k tok/s | reste ~5 min
pas 1920/3052 | loss 3.4471 | lr 7.96e-04 | 123.2k tok/s | reste ~5 min
pas 1980/3052 | loss 3.2539 | lr 7.42e-04 | 123.4k tok/s | reste ~5 min
pas 2000 | BPB validation (1/4 du jeu) : 1.4008
pas 2040/3052 | loss 3.2648 | lr 6.89e-04 | 123.0k tok/s | reste ~4 min
pas 2100/3052 | loss 3.2406 | lr 6.38e-04 | 122.9k tok/s | reste ~4 min
pas 216

In [8]:
%cd /kaggle/working
!zip ablation-extra-results.zip runs/ablation-extra/manifest.json runs/ablation-extra/environment.json runs/ablation-extra/*/result.json


/kaggle/working
  adding: runs/ablation-extra/manifest.json (deflated 69%)
  adding: runs/ablation-extra/environment.json (deflated 23%)
  adding: runs/ablation-extra/pii/result.json (deflated 58%)
  adding: runs/ablation-extra/quality_pii/result.json (deflated 58%)
  adding: runs/ablation-extra/quality/result.json (deflated 58%)
